# Setup Data: Load Silver Tables

**Run on dedicated or serverless compute. Run this notebook once, before `01_required_setup`, if the Silver Delta tables have not already been loaded by your workshop admin.**

This notebook builds the five base (Silver) tables the rest of the workshop reads from. It mirrors `enrichment-pipeline/upload_and_create_tables.sh`, but as a self-contained notebook: it fetches the committed CSVs straight from the public GitHub repo (the same approach as `aircraft-graphrag/notebooks/01_etl_to_neo4j.ipynb`), writes them to a Unity Catalog Volume, creates the tables with column-level comments, and loads the data.

## What it does

1. Create the schema and Volume
2. Create the five Silver tables with Unity Catalog column comments (the primary signal Genie uses)
3. Download the committed CSVs and `ground_truth.json` from GitHub into the Volume
4. `INSERT OVERWRITE` each table from the uploaded CSVs
5. Verify row counts

## Tables produced

`accounts`, `merchants`, `transactions`, `account_links`, `account_labels` in `` `graph-on-databricks`.`graph-enriched-schema` `` — exactly what `03_neo4j_ingest` reads.

## Step 0: Configuration

`CATALOG` / `SCHEMA` / `VOLUME` are the Silver targets the downstream notebooks read from. `GITHUB_BASE` is the raw-content URL for the committed `finance-genie/data/` directory; change `GITHUB_REF` if you are loading from a branch other than `main`.

In [ ]:
CATALOG = "graph-on-databricks"
SCHEMA  = "graph-enriched-schema"
VOLUME  = "graph-enriched-volume"

GITHUB_REF  = "main"
GITHUB_BASE = (
    "https://raw.githubusercontent.com/neo4j-partners/"
    f"graph-on-databricks/{GITHUB_REF}/finance-genie/data"
)

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

CSV_FILES = [
    "accounts.csv",
    "merchants.csv",
    "transactions.csv",
    "account_links.csv",
    "account_labels.csv",
]

print(f"Target tables: `{CATALOG}`.`{SCHEMA}`.<table>")
print(f"Volume:        {VOLUME_PATH}")
print(f"Source:        {GITHUB_BASE}")

## Step 1: Create the schema and Volume

`CREATE SCHEMA`/`CREATE VOLUME` are idempotent, so this is safe to re-run.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`")
print(f"  [OK] schema `{CATALOG}`.`{SCHEMA}`")
print(f"  [OK] volume {VOLUME_PATH}")

## Step 2: Create the Silver tables

`CREATE OR REPLACE TABLE` defines column types and the Unity Catalog column comments Genie reads to understand the data. The comments live in this DDL, so they survive every re-run. Informational `FOREIGN KEY ... RELY` constraints are added afterward as the declared-relationship signal for downstream tooling (Unity Catalog does not enforce them).

In [ ]:
CREATE_TABLES = [
    f"""
    CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.accounts (
        account_id   BIGINT  NOT NULL COMMENT 'Unique account identifier (primary key)',
        account_hash STRING           COMMENT 'Anonymized account identifier derived from the original account number',
        account_name STRING           COMMENT 'Account holder name: a person name for checking/savings accounts, a company name for business accounts',
        account_type STRING           COMMENT 'Account category: checking, savings, or business',
        region       STRING           COMMENT 'Geographic region where the account was opened',
        balance      DOUBLE           COMMENT 'Current account balance in USD',
        opened_date  DATE             COMMENT 'Date the account was opened',
        holder_age   INT              COMMENT 'Age of the account holder in years',
        CONSTRAINT accounts_pk PRIMARY KEY (account_id) RELY
    ) USING DELTA
    COMMENT 'Account dimension - one row per account holder'
    """,
    f"""
    CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.merchants (
        merchant_id   BIGINT  NOT NULL COMMENT 'Unique merchant identifier (primary key)',
        merchant_name STRING           COMMENT 'Merchant business name',
        category      STRING           COMMENT 'Merchant business category (e.g., retail, food, entertainment)',
        region        STRING           COMMENT 'Geographic region where the merchant operates',
        CONSTRAINT merchants_pk PRIMARY KEY (merchant_id) RELY
    ) USING DELTA
    COMMENT 'Merchant dimension - one row per merchant'
    """,
    f"""
    CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.transactions (
        txn_id        BIGINT     NOT NULL COMMENT 'Unique transaction identifier (primary key)',
        account_id    BIGINT              COMMENT 'Account that initiated the payment (foreign key to accounts.account_id)',
        merchant_id   BIGINT              COMMENT 'Merchant that received the payment (foreign key to merchants.merchant_id)',
        amount        DOUBLE              COMMENT 'Transaction amount in USD',
        txn_timestamp TIMESTAMP           COMMENT 'Timestamp when the transaction occurred',
        txn_hour      INT                 COMMENT 'Hour of day (0-23) when the transaction occurred',
        CONSTRAINT transactions_pk PRIMARY KEY (txn_id) RELY
    ) USING DELTA
    COMMENT 'Transaction fact table - one row per account-to-merchant payment event'
    """,
    f"""
    CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.account_links (
        link_id            BIGINT     NOT NULL COMMENT 'Unique transfer event identifier (primary key)',
        src_account_id     BIGINT              COMMENT 'Account that sent the transfer (foreign key to accounts.account_id)',
        dst_account_id     BIGINT              COMMENT 'Account that received the transfer (foreign key to accounts.account_id)',
        amount             DOUBLE              COMMENT 'Transfer amount in USD',
        transfer_timestamp TIMESTAMP           COMMENT 'Timestamp when the transfer occurred',
        CONSTRAINT account_links_pk PRIMARY KEY (link_id) RELY
    ) USING DELTA
    COMMENT 'Account-to-account transfer graph - one row per directed transfer event'
    """,
    f"""
    CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.account_labels (
        account_id BIGINT  NOT NULL COMMENT 'Account identifier (foreign key to accounts.account_id)',
        is_fraud   BOOLEAN          COMMENT 'Ground-truth fraud label: true if the account is a confirmed fraud ring member',
        CONSTRAINT account_labels_pk PRIMARY KEY (account_id) RELY
    ) USING DELTA
    COMMENT 'Ground-truth fraud labels - one row per account'
    """,
]

for ddl in CREATE_TABLES:
    spark.sql(ddl)
print(f"  [OK] created 5 tables in `{CATALOG}`.`{SCHEMA}`")

# Informational FOREIGN KEY (RELY) constraints - added after all tables exist.
FK_CONSTRAINTS = [
    f"ALTER TABLE `{CATALOG}`.`{SCHEMA}`.transactions ADD CONSTRAINT transactions_account_fk FOREIGN KEY (account_id) REFERENCES `{CATALOG}`.`{SCHEMA}`.accounts (account_id) RELY",
    f"ALTER TABLE `{CATALOG}`.`{SCHEMA}`.transactions ADD CONSTRAINT transactions_merchant_fk FOREIGN KEY (merchant_id) REFERENCES `{CATALOG}`.`{SCHEMA}`.merchants (merchant_id) RELY",
    f"ALTER TABLE `{CATALOG}`.`{SCHEMA}`.account_links ADD CONSTRAINT account_links_src_fk FOREIGN KEY (src_account_id) REFERENCES `{CATALOG}`.`{SCHEMA}`.accounts (account_id) RELY",
    f"ALTER TABLE `{CATALOG}`.`{SCHEMA}`.account_links ADD CONSTRAINT account_links_dst_fk FOREIGN KEY (dst_account_id) REFERENCES `{CATALOG}`.`{SCHEMA}`.accounts (account_id) RELY",
    f"ALTER TABLE `{CATALOG}`.`{SCHEMA}`.account_labels ADD CONSTRAINT account_labels_account_fk FOREIGN KEY (account_id) REFERENCES `{CATALOG}`.`{SCHEMA}`.accounts (account_id) RELY",
]
for fk in FK_CONSTRAINTS:
    spark.sql(fk)
print("  [OK] added 5 foreign-key (RELY) constraints")

## Step 3: Download data into the Volume

Fetch each committed CSV and `ground_truth.json` from GitHub raw and write them into the Volume. Unity Catalog Volumes support the local file API, so a plain `open(...)` write lands the file under `/Volumes/...`.

In [ ]:
import urllib.request

for filename in CSV_FILES + ["ground_truth.json"]:
    url  = f"{GITHUB_BASE}/{filename}"
    dest = f"{VOLUME_PATH}/{filename}"
    with urllib.request.urlopen(url) as response:
        data = response.read()
    with open(dest, "wb") as f:
        f.write(data)
    print(f"  [OK] {filename:<20} ({len(data):>10,} bytes) -> {dest}")

## Step 4: Load data into the tables

`INSERT OVERWRITE` replaces all rows without touching the schema or column comments. Every column is read as `STRING` from the CSV (`inferSchema => 'false'`) and cast explicitly, so types match the DDL exactly.

In [ ]:
LOADS = {
    "accounts": f"""
        INSERT OVERWRITE `{CATALOG}`.`{SCHEMA}`.accounts
        SELECT CAST(account_id AS BIGINT), account_hash, account_name, account_type,
               region, CAST(balance AS DOUBLE), CAST(opened_date AS DATE), CAST(holder_age AS INT)
        FROM read_files('{VOLUME_PATH}/accounts.csv', format => 'csv', header => 'true', inferSchema => 'false',
            schema => 'account_id STRING, account_hash STRING, account_name STRING, account_type STRING, region STRING, balance STRING, opened_date STRING, holder_age STRING')
    """,
    "merchants": f"""
        INSERT OVERWRITE `{CATALOG}`.`{SCHEMA}`.merchants
        SELECT CAST(merchant_id AS BIGINT), merchant_name, category, region
        FROM read_files('{VOLUME_PATH}/merchants.csv', format => 'csv', header => 'true', inferSchema => 'false',
            schema => 'merchant_id STRING, merchant_name STRING, category STRING, region STRING')
    """,
    "transactions": f"""
        INSERT OVERWRITE `{CATALOG}`.`{SCHEMA}`.transactions
        SELECT CAST(txn_id AS BIGINT), CAST(account_id AS BIGINT), CAST(merchant_id AS BIGINT),
               CAST(amount AS DOUBLE), CAST(txn_timestamp AS TIMESTAMP), CAST(txn_hour AS INT)
        FROM read_files('{VOLUME_PATH}/transactions.csv', format => 'csv', header => 'true', inferSchema => 'false',
            schema => 'txn_id STRING, account_id STRING, merchant_id STRING, amount STRING, txn_timestamp STRING, txn_hour STRING')
    """,
    "account_links": f"""
        INSERT OVERWRITE `{CATALOG}`.`{SCHEMA}`.account_links
        SELECT CAST(link_id AS BIGINT), CAST(src_account_id AS BIGINT), CAST(dst_account_id AS BIGINT),
               CAST(amount AS DOUBLE), CAST(transfer_timestamp AS TIMESTAMP)
        FROM read_files('{VOLUME_PATH}/account_links.csv', format => 'csv', header => 'true', inferSchema => 'false',
            schema => 'link_id STRING, src_account_id STRING, dst_account_id STRING, amount STRING, transfer_timestamp STRING')
    """,
    "account_labels": f"""
        INSERT OVERWRITE `{CATALOG}`.`{SCHEMA}`.account_labels
        SELECT CAST(account_id AS BIGINT),
               CAST(CASE WHEN lower(is_fraud) = 'true' THEN 'true' ELSE 'false' END AS BOOLEAN)
        FROM read_files('{VOLUME_PATH}/account_labels.csv', format => 'csv', header => 'true', inferSchema => 'false',
            schema => 'account_id STRING, is_fraud STRING')
    """,
}

for table, sql in LOADS.items():
    spark.sql(sql)
    print(f"  [OK] loaded {table}")

## Step 5: Verify row counts

In [ ]:
for table in ["accounts", "merchants", "transactions", "account_links", "account_labels"]:
    n = spark.table(f"`{CATALOG}`.`{SCHEMA}`.{table}").count()
    print(f"  {table:<16} {n:>10,} rows")

## Next steps

The Silver tables are loaded. Proceed to **`01_required_setup`** to store your Neo4j credentials and Genie Space IDs, then continue through the workshop.